<div style="font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif;max-width:900px;margin:0 auto;border-radius:16px;overflow:hidden;box-shadow:0 4px 20px rgba(0,0,0,0.12);border:1px solid #e2e2e2;">

<div style="background:linear-gradient(135deg,#002855 0%,#004b8d 55%,#0077c8 100%);padding:28px 20px 22px 20px;text-align:center;">
<img src="./img/ITESOLogo.png" alt="ITESO" width="260" style="margin-bottom:10px;">
<div style="color:#ffffff;font-size:15px;font-weight:600;letter-spacing:0.5px;text-transform:uppercase;opacity:0.9;">
Departamento de Electrónica, Sistemas e Informática
</div>
</div>

<div style="background-color:#ffffff;padding:26px 30px 30px 30px;text-align:center;">

<div style="color:#002855;font-size:26px;font-weight:800;margin-bottom:6px;">
Big Data Analysis
</div>

<div style="display:inline-block;background-color:#eaf4fb;color:#0077c8;font-size:13px;font-weight:700;padding:4px 14px;border-radius:20px;letter-spacing:0.5px;margin-bottom:22px;">
Autumn 2026
</div>


<hr style="border:none;border-top:2px solid #f0f0f0;margin:0 0 22px 0;">

<div style="background-color:#f7fafd;border-left:5px solid #0077c8;border-radius:8px;padding:14px 18px;text-align:left;margin-bottom:18px;">
<div style="font-size:12px;color:#7a7a7a;font-weight:600;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:4px;">
Session 14
</div>
<div style="font-size:19px;color:#002855;font-weight:700;">
Big Data - Storage Solutions: SQL Databases
</div>
</div>

<div style="font-size:14px;color:#444;margin-top:20px;">
<span style="font-weight:700;color:#002855;">Profesor:</span> Pablo Camarillo Ramírez
</div>

</div>
</div>

# Create SparkSession

In [ ]:
!ls /opt/spark/work-dir/notebooks/jars/postgresql-42.7.13.jar

In [ ]:
from pcamarillor.spark_utils import SparkUtils

su = SparkUtils("Examples on storage solutions with PosgreSQL",
                "local[*]",
                spark_jars="/opt/spark/work-dir/notebooks/jars/postgresql-42.7.13.jar")
su.spark

# Create DataFrames

In [ ]:
!pwd
!ls data/movies/

In [ ]:
import pyspark.sql.functions as F
movies_columns = [("MovieID", "int"),
                  ("Title", "string"),
                  ("Genre", "string"),
                  ("ReleaseYear", "int"),
                  ("ReleaseDate_Str", "string"),
                  ("Country", "string"),
                  ("BudgetUSD", "float"),
                  ("US_BoxOfficeUSD", "float"),
                  ("Global_BoxOfficeUSD", "float"),
                  ("Opening_Day_SalesUSD", "float"),
                  ("One_Week_SalesUSD", "float"),
                  ("IMDbRating", "float"),
                  ("RottenTomatoesScore", "float"),
                  ("NumVotesIMDb", "int"),
                  ("NumVotesRT", "int"),
                  ("Director", "string"),
                  ("LeadActor", "string")]

movies_schema = SparkUtils.generate_schema(movies_columns)

movies_date_str_df = (su.spark.read 
                .option("header", "true")
                .schema(movies_schema)
                .csv("/opt/spark/work-dir/data/movies/"))

movies_df = movies_date_str_df.withColumn("ReleaseDate", F.to_date("ReleaseDate_Str", "dd-MM-yyyy")).drop("ReleaseDate_Str")
movies_df = movies_df.groupBy("Genre").count().orderBy(F.col("count"), ascending=False)
movies_df.show()

# Write data to a PostgreSQL DB

## Install PostgreSQL with Docker

    docker run -d --name postgres-iteso --network spark-cluster_default -e POSTGRES_PASSWORD=Admin@1234 postgres

## Create a new DB
    CREATE DATABASE <db name>;

In [ ]:
jdbc_url = "jdbc:postgresql://postgres-iteso:5432/dbiteso"
table_name = "grouped_movies"

(movies_df.write
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", table_name)
    .option("user", "postgres")
    .option("password", "Admin@1234")
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite") # "append", "overwrite", "ignore", "error", "errorifexists", and "default"
    .save())

print("DataFrame successfully written into a PosgreSQL DB !")

# Explore commands

    \l                 -- list databases
    \c <database>      -- switch to a database
    \dn                -- list schemas
    \dt                -- list tables in the current schema
    \dt *.*            -- list tables in all schemas
    \d <table_name>    -- show columns, types, indexes and constraints
    \q                 -- quit

# Read data from a PosgreSQL DB

In [ ]:
jdbc_url = "jdbc:postgresql://postgres-iteso:5432/dbiteso"
db_properties = {
      "user": "postgres",      
      "password": "Admin@1234",
      "driver": "org.postgresql.Driver"
  }

df = (su.spark.read
    .jdbc(url=jdbc_url, table=table_name, properties=db_properties))

df.printSchema()
df.show()

In [7]:
su.spark.stop()